# Modul 6 — Filter Spasial
### Low Pass Filter, High Pass Filter, Point Detection, Line Detection, Edge Detection

**Pengolahan Citra dan Visi Komputer — Jurusan Teknologi Informasi**

Notebook ini adalah pengerjaan **lengkap** Modul 6 (seluruh bagian Praktikum, Pertanyaan, dan
Tugas) pada lingkungan **Kaggle**. Tidak ada `google.colab`, `drive.mount`, maupun
`files.upload()`; seluruh citra diakses melalui `BASE`/`WEEK` yang otomatis menunjuk ke
`/kaggle/input/.../pcvk-images/week6` di Kaggle atau folder `datasets/week6` saat dijalankan lokal.

**Isi notebook**

| Bagian | Cakupan |
|---|---|
| D. Praktikum | Fungsi konvolusi manual (nested loop, tanpa `cv.filter2D`) + seluruh kernel filter dasar |
| Kernel manual | Average 3×3, Low Pass, High Pass, Sharpen, Emboss, Left Sobel, Canny-like, Point Detection, Line Detection (H/V/+45/−45), Edge Detection, Gaussian 21×21 |
| E. Filter Library & Modern | Percobaan 1–7 (filter2D, Bilateral & Guided, Feature Map CNN, Beauty/Vintage, Cartoon, Night, Morning+Fog) |
| Tugas Praktikum | (1) Salt & Pepper: Mean vs Median, (2) Sharpening 3×3 vs 5×5, (3) Fungsi `kartun(image)` |

Seluruh citra keluaran diberi judul. Jawaban Pertanyaan dan analisis Tugas ditulis sebagai sel markdown.

In [ ]:
# ==== KONFIGURASI LINGKUNGAN (jalan di Kaggle maupun lokal) ====
import os, sys, glob, math
import numpy as np
import cv2 as cv
import matplotlib.pyplot as plt

def _find_base():
    for p in ('/kaggle/input/pcvk-images',
              '/kaggle/input/datasets/singgihwahyupermana/pcvk-images'):
        if os.path.isdir(p):
            return p
    _hit = glob.glob('/kaggle/input/**/pcvk-images', recursive=True)
    if _hit:
        return _hit[0]
    d = os.getcwd()
    for _ in range(6):
        if all(os.path.isdir(os.path.join(d, f'week{i}')) for i in (1, 2, 3)):
            return d
        d = os.path.dirname(d)
    return None

BASE = _find_base()
WEEK = os.path.join(BASE, 'week6')   # <-- ganti angka minggu sesuai notebook

def show(img, title='', cmap=None, figsize=(6, 4)):
    plt.figure(figsize=figsize)
    if img.ndim == 3:
        plt.imshow(cv.cvtColor(img, cv.COLOR_BGR2RGB))
    else:
        plt.imshow(img, cmap=cmap or 'gray')
    plt.title(title); plt.axis('off'); plt.show()

# Shim cv2_imshow (Colab) -> Kaggle
try:
    from google.colab.patches import cv2_imshow
except Exception:
    def cv2_imshow(img):
        show(img)

print('BASE :', BASE)
print('WEEK :', WEEK)
print('files:', sorted(os.listdir(WEEK)) if WEEK and os.path.isdir(WEEK) else 'MISSING')

In [ ]:
OUT = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
print('OUT :', OUT)

---
## D. Praktikum — Fungsi Konvolusi Manual

### D.1 Algoritma Konvolusi

Konvolusi 2D antara citra $f(x,y)$ dan kernel $h(x,y)$ didefinisikan sebagai:

$$g(x,y) = f(x,y) \circledast h(x,y) = \sum_{a=0}^{M-1}\sum_{b=0}^{N-1} f(a,b)\, h(x-a,\, y-b)$$

Kernel digeser (sliding) ke seluruh posisi citra. Pada setiap posisi dihitung **jumlah perkalian
titik** (*sum of products*) antara koefisien kernel dengan piksel tetangga. Fungsi di bawah ini
mengimplementasikan persamaan tersebut secara **manual dengan nested loop**, sesuai pseudocode modul:

```
For x = 0 to image_width-1
  For y = 0 to image_height-1
    z(x,y) = 0
    For k1 = 0 to filter_width-1
      For k2 = 0 to filter_height-1
        z(x,y) = z(x,y) + H(k1,k2) * Img_pixel(x+k1, y+k2)
```

**Penanganan piksel pinggir (efek "menggantung")** — modul menyebut tiga solusi, dan ketiganya
diimplementasikan melalui parameter `padding`:

| `padding` | Perilaku piksel pinggir |
|---|---|
| `'zero'` | piksel di luar citra dianggap **0** (constant padding) |
| `'replicate'` | piksel luar **disalin dari tepi terdekat** (edge duplication) |
| `'same'` | piksel pinggir **tidak dikonvolusi**, nilainya tetap sama seperti citra asal |

> Catatan: tidak ada pemanggilan `cv.filter2D`, `cv.blur`, `scipy.signal.convolve`, atau rutin
> konvolusi library lain di dalam fungsi ini.

In [ ]:
# ---- Fungsi konvolusi manual (TANPA library konvolusi apa pun) ----
def convolve2d(image, kernel, padding='zero'):
    """Konvolusi 2D manual dengan nested loop.

    padding : 'zero'      -> piksel di luar citra bernilai 0
              'replicate' -> piksel di luar citra disalin dari tepi terdekat
              'same'      -> piksel pinggir TIDAK dikonvolusi (nilai asli dipertahankan)
    """
    img = np.asarray(image, dtype=np.float64)
    if img.ndim != 2:
        raise ValueError('fungsi ini untuk citra grayscale (1 kanal)')
    k = np.asarray(kernel, dtype=np.float64)
    kh, kw = k.shape
    ph, pw = kh // 2, kw // 2
    H, W = img.shape

    if padding == 'replicate':
        padded = np.pad(img, ((ph, ph), (pw, pw)), mode='edge')
    else:  # 'zero' dan 'same' memakai padding nol pada area yang benar-benar dihitung
        padded = np.pad(img, ((ph, ph), (pw, pw)), mode='constant', constant_values=0)

    out = np.zeros((H, W), dtype=np.float64)
    for i in range(H):               # geser baris citra
        for j in range(W):           # geser kolom citra
            total = 0.0
            for ki in range(kh):     # baris kernel
                for kj in range(kw): # kolom kernel
                    total += k[ki, kj] * padded[i + ki, j + kj]
            out[i, j] = total

    if padding == 'same':            # pinggir dikembalikan seperti citra asal
        out[:ph, :]  = img[:ph, :]
        out[-ph:, :] = img[-ph:, :]
        out[:, :pw]  = img[:, :pw]
        out[:, -pw:] = img[:, -pw:]
    return out

def to_uint8(x, absolute=False):
    """Normalisasi hasil konvolusi (yang bisa bernilai negatif) menjadi uint8 untuk ditampilkan."""
    x = np.abs(x) if absolute else x
    return np.clip(x, 0, 255).astype(np.uint8)

print('convolve2d siap digunakan.')

### D.2 Verifikasi Numerik pada Contoh Modul (matriks 5×5)

Modul memberikan contoh citra $f(x,y)$ berukuran 5×5 dan kernel:

$$f = \begin{bmatrix} 4&4&3&5&4\\ 6&6&5&5&2\\ 5&6&6&6&2\\ 6&7&5&5&3\\ 3&5&2&4&4 \end{bmatrix},
\qquad h = \begin{bmatrix} 0&-1&0\\ -1&4&-1\\ 0&-1&0 \end{bmatrix}$$

Nilai hasil konvolusi pada pusat (baris 2, kolom 2) dihitung tangan:
$g(2,2) = 4(6) - 1(5) - 1(6) - 1(5) - 1(6) = 24 - 22 = 2$.
Sel di bawah memverifikasi bahwa `convolve2d` menghasilkan angka yang sama.

In [ ]:
# ---- Uji numerik: contoh 5x5 dari modul ----
f = np.array([[4, 4, 3, 5, 4],
              [6, 6, 5, 5, 2],
              [5, 6, 6, 6, 2],
              [6, 7, 5, 5, 3],
              [3, 5, 2, 4, 4]], dtype=np.float64)
h = np.array([[0, -1, 0],
              [-1, 4, -1],
              [0, -1, 0]], dtype=np.float64)

hasil = convolve2d(f, h, padding='zero')
print('citra input (5x5):'); print(f.astype(int))
print('kernel (3x3):'); print(h.astype(int))
print('hasil konvolusi:'); print(hasil)
print('g(2,2) hasil program =', hasil[2, 2], '| hitung tangan =', 4*6 - 5 - 6 - 5 - 6)
assert hasil[2, 2] == 4*6 - 5 - 6 - 5 - 6, 'verifikasi numerik gagal!'
print('Verifikasi numerik BERHASIL.')

### D.3 Memuat Citra dan Mengubah ke Grayscale

Modul meminta citra di-*load* lalu dikonversi ke citra keabuan. Untuk bagian konvolusi manual
(proses per-piksel dengan Python murni) citra diperkecil ke 200×200 agar eksekusi tetap cepat,
sementara citra ukuran penuh dipakai pada bagian library (Percobaan 1–7).

In [ ]:
img_color = cv.imread(os.path.join(WEEK, 'peppers.jpg'))
assert img_color is not None, 'peppers.jpg tidak ditemukan di ' + str(WEEK)
gray_full = cv.cvtColor(img_color, cv.COLOR_BGR2GRAY)

# versi kecil khusus demonstrasi konvolusi manual
gray = cv.resize(gray_full, (200, 200), interpolation=cv.INTER_AREA)
print('ukuran citra penuh :', gray_full.shape)
print('ukuran citra demo  :', gray.shape)

show(img_color, 'Citra Asli - peppers.jpg (warna)')
show(gray, 'Citra Grayscale 200x200 (untuk konvolusi manual)')

### D.4 Definisi Kernel Filter

Kernel-kernel berikut diambil dari modul (Bagian C.3 dan tabel Praktikum). Setiap kernel
dinormalisasi dengan membaginya dengan jumlah elemen bila tujuannya *smoothing* (agar
kecerahan rata-rata citra tidak berubah).

| Filter | Kernel |
|---|---|
| Average 3×3 | $\frac{1}{9}$ semua elemen bernilai 1 |
| Low Pass | $\frac{1}{12}\begin{bmatrix}1&1&1\\1&4&1\\1&1&1\end{bmatrix}$ |
| High Pass | $\begin{bmatrix}-1&0&1\\-1&0&3\\-3&0&1\end{bmatrix}$ |
| Sharpen | $\begin{bmatrix}0&-1&0\\-1&5&-1\\0&-1&0\end{bmatrix}$ |
| Emboss | $\begin{bmatrix}-2&-1&0\\-1&1&1\\0&1&2\end{bmatrix}$ |
| Left Sobel | $\begin{bmatrix}1&0&-1\\2&0&-2\\1&0&-1\end{bmatrix}$ |
| Canny-like | $\begin{bmatrix}-1&-1&-1\\-1&8&-1\\-1&-1&-1\end{bmatrix}$ |
| Point Detection | $\begin{bmatrix}-1&-1&-1\\-1&8&-1\\-1&-1&-1\end{bmatrix}$ |
| Line H / V / +45 / −45 | kernel 3×3 arah horizontal, vertikal, diagonal |
| Edge Detection | $\begin{bmatrix}0&1&0\\1&-4&1\\0&1&0\end{bmatrix}$ |

In [ ]:
KERNELS = {
    'Average 3x3':   np.ones((3, 3), np.float64) / 9.0,
    'Low Pass':      np.array([[1, 1, 1], [1, 4, 1], [1, 1, 1]], np.float64) / 12.0,
    'High Pass':     np.array([[-1, 0, 1], [-1, 0, 3], [-3, 0, 1]], np.float64),
    'Sharpen':       np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], np.float64),
    'Emboss':        np.array([[-2, -1, 0], [-1, 1, 1], [0, 1, 2]], np.float64),
    'Left Sobel':    np.array([[1, 0, -1], [2, 0, -2], [1, 0, -1]], np.float64),
    'Canny-like':    np.array([[-1, -1, -1], [-1, 8, -1], [-1, -1, -1]], np.float64),
    'Point Detect':  np.array([[-1, -1, -1], [-1, 8, -1], [-1, -1, -1]], np.float64),
    'Line H':        np.array([[-1, -1, -1], [2, 2, 2], [-1, -1, -1]], np.float64),
    'Line V':        np.array([[-1, 2, -1], [-1, 2, -1], [-1, 2, -1]], np.float64),
    'Line +45':      np.array([[-1, -1, 2], [-1, 2, -1], [2, -1, -1]], np.float64),
    'Line -45':      np.array([[2, -1, -1], [-1, 2, -1], [-1, -1, 2]], np.float64),
    'Edge Detect':   np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], np.float64),
}

# kernel yang hasilnya berupa respons tepi (nilai negatif -> ditampilkan sebagai |nilai|)
ABS_FILTERS = {'High Pass', 'Left Sobel', 'Canny-like', 'Point Detect',
               'Line H', 'Line V', 'Line +45', 'Line -45', 'Edge Detect'}

print('jumlah kernel:', len(KERNELS))
for nama, k in KERNELS.items():
    print(f'{nama:12s} ukuran={k.shape} jumlah_koefisien={k.sum():7.3f}')

### D.5 Menerapkan Seluruh Filter dengan `convolve2d`

Semua kernel dari D.4 dikonvolusikan pada citra grayscale memakai fungsi manual. Filter yang
menghasilkan respons tepi (koefisien negatif) ditampilkan sebagai nilai absolut $|g(x,y)|$ agar
tepi terlihat sebagai garis terang.

In [ ]:
def show_grid(images, titles, ncols=4, figsize=(15, 9)):
    """Menampilkan banyak citra grayscale dalam satu grid."""
    n = len(images)
    nrows = int(math.ceil(n / ncols))
    plt.figure(figsize=figsize)
    for idx, (im, t) in enumerate(zip(images, titles), 1):
        plt.subplot(nrows, ncols, idx)
        plt.imshow(im, cmap='gray', vmin=0, vmax=255)
        plt.title(t, fontsize=9); plt.axis('off')
    plt.tight_layout(); plt.show()

hasil_kernel = {}
for nama, k in KERNELS.items():
    r = convolve2d(gray, k, padding='zero')
    hasil_kernel[nama] = to_uint8(r, absolute=(nama in ABS_FILTERS))

show_grid([gray] + list(hasil_kernel.values()),
          ['Citra Asli'] + list(hasil_kernel.keys()), ncols=4, figsize=(15, 12))

### D.6 Pengaruh Opsi Padding pada Piksel Pinggir

Tiga pendekatan penanganan pinggir diuji memakai filter Sharpen. Perbedaan paling jelas ada di
tepi citra: `zero` menimbulkan garis gelap/terang buatan di border, `replicate` menirukan warna
tetangga terdekat sehingga border lebih natural, sedangkan `same` membiarkan border apa adanya
(tidak dikonvolusi).

In [ ]:
pad_zero = convolve2d(gray, KERNELS['Sharpen'], padding='zero')
pad_repl = convolve2d(gray, KERNELS['Sharpen'], padding='replicate')
pad_same = convolve2d(gray, KERNELS['Sharpen'], padding='same')

# perbesar tepi kiri-atas untuk memperlihatkan efek border
crop = (slice(0, 40), slice(0, 40))
show_grid([to_uint8(pad_zero)[crop], to_uint8(pad_repl)[crop], to_uint8(pad_same)[crop]],
          ['Sharpen padding=zero (crop 40x40)',
           'Sharpen padding=replicate (crop 40x40)',
           'Sharpen padding=same (crop 40x40)'], ncols=3, figsize=(12, 4))

print('Baris paling atas (0:1) hasil sharpen:')
print(' zero      :', to_uint8(pad_zero)[0, :8])
print(' replicate :', to_uint8(pad_repl)[0, :8])
print(' same      :', to_uint8(pad_same)[0, :8], '(= piksel asli, tidak dikonvolusi)')

### D.7 Kernel Gaussian 21×21 (dibuat dengan `cv.getGaussianKernel`)

Modul meminta pembuatan kernel Gaussian 21×21 dengan:

```python
sigma = math.sqrt(kernel_size)
gaussian_kernel = cv.getGaussianKernel(kernel_size, sigma)
gauss_kernel = gaussian_kernel @ gaussian_kernel.transpose()
```

Kernel 1D hasil `cv.getGaussianKernel` bersifat **separable**; hasil *outer product*-nya adalah
kernel Gaussian 2D 21×21. Karena 21×21 berarti 441 operasi per piksel, konvolusi manual di sini
dijalankan pada citra 140×140 agar tetap efisien (algoritma tetap sama, hanya ukuran masukan yang
diperkecil).

Sel terakhir membandingkan hasil manual dengan `cv.filter2D` memakai **penanganan border yang sama**
(`BORDER_REPLICATE`); bila border diperlakukan berbeda, selisih hanya muncul di piksel pinggir.

In [ ]:
kernel_size = 21
sigma = math.sqrt(kernel_size)
gaussian_kernel = cv.getGaussianKernel(kernel_size, sigma)
gauss_kernel = gaussian_kernel @ gaussian_kernel.transpose()
print('bentuk gauss_kernel :', gauss_kernel.shape)
print('jumlah koefisien    :', gauss_kernel.sum(), '(idealnya ~1.0)')

gray_small = cv.resize(gray_full, (140, 140), interpolation=cv.INTER_AREA)
blur21 = convolve2d(gray_small, gauss_kernel, padding='replicate')

show_grid([gray_small, to_uint8(blur21)],
          ['Asli 140x140', 'Gaussian Blur 21x21 (konvolusi manual)'],
          ncols=2, figsize=(9, 4))

# perbandingan dengan hasil library (untuk membuktikan kebenaran implementasi manual).
# cv.filter2D memakai BORDER_REFLECT_101 secara default, jadi borderType disamakan
# dengan padding 'replicate' milik convolve2d agar perbandingannya adil.
blur21_lib = cv.filter2D(gray_small.astype(np.float64), -1, gauss_kernel,
                         borderType=cv.BORDER_REPLICATE)
print('selisih maksimum manual vs cv.filter2D :',
      float(np.max(np.abs(blur21 - blur21_lib))))
print('-> implementasi manual BENAR (selisih praktis nol bila penanganan border sama)')

### Jawaban — Pengaruh Ukuran dan Jenis Kernel

- **Average 3×3 dan Low Pass** bertindak sebagai *low pass filter*: frekuensi tinggi (detail, tepi,
  noise) ditekan sehingga citra tampak halus. Low Pass (bobot pusat 4) lebih lembut dari Average
  karena menekankan piksel pusat.
- **High Pass, Sharpen, dan Edge Detection** bertindak sebagai *high pass filter*: komponen
  frekuensi rendah (warna rata) dihilangkan, menyisakan tepi. Sharpen menambahkan kembali tepi ke
  citra asal (jumlah kernel = 1) sehingga tampak lebih tajam tanpa kehilangan latar.
- **Left Sobel** menonjolkan tepi yang menghadap satu arah (horizontal), sehingga tepi vertikal
  terlihat sebagai gradien terang–gelap.
- **Emboss** memberi efek timbul 3D dengan mempertahankan komponen warna pada arah tertentu
  (jumlah kernel = 3, hasil digeser 128 bila ingin tampak abu-abu).
- **Point Detection / Canny-like** (Laplacian 8-tetangga) merespons titik dan tepi tajam di segala
  arah, **Line Detection** merespons garis sesuai orientasi kernelnya (H, V, ±45°).
- **Gaussian 21×21** menghasilkan blur paling kuat dan paling halus karena bobotnya menurun
  mengikuti distribusi Gaussian; hasil manual identik dengan `cv.filter2D` (selisih maksimum ≈ 0).

---
## E. Filter Library dan Filter Modern

Pada bagian ini filter diterapkan memakai fungsi library OpenCV dan kombinasi filter tradisional,
sesuai Percobaan 1–7 pada modul.

### Percobaan 1 — Filter Gaussian, Sharpen, dan Canny dengan `cv.filter2D` (Citra RGB)

`cv.filter2D` bekerja pada citra berkanal banyak (RGB/BGR) dengan mengonvolusi setiap kanal
menggunakan kernel yang sama. Fungsi `show_side_by_side` menampilkan hasil berdampingan dengan
citra asli.

In [ ]:
def show_side_by_side(images, titles, figsize=(16, 5)):
    """Menampilkan beberapa citra berdampingan (kiri ke kanan)."""
    n = len(images)
    plt.figure(figsize=figsize)
    for i, (im, t) in enumerate(zip(images, titles), 1):
        plt.subplot(1, n, i)
        if im.ndim == 3:
            plt.imshow(cv.cvtColor(im, cv.COLOR_BGR2RGB))
        else:
            plt.imshow(im, cmap='gray')
        plt.title(t); plt.axis('off')
    plt.tight_layout(); plt.show()

img = cv.imread(os.path.join(WEEK, 'peppers.jpg'))
assert img is not None

# kernel Gaussian 9x9 (separable: outer product dari kernel 1D)
g1d = cv.getGaussianKernel(9, 2.0)
K_gauss = g1d @ g1d.transpose()
# kernel sharpen 3x3
K_sharp = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], np.float32)

gauss_rgb  = cv.filter2D(img, -1, K_gauss)     # low pass pada RGB
sharp_rgb  = cv.filter2D(img, -1, K_sharp)     # high pass / sharpen pada RGB
gray_rgb   = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
canny_rgb  = cv.Canny(gray_rgb, 100, 200)      # deteksi tepi Canny

show_side_by_side([img, gauss_rgb, sharp_rgb, canny_rgb],
                  ['Asli', 'Gaussian (filter2D)', 'Sharpen (filter2D)', 'Canny (edge)'],
                  figsize=(18, 5))

cv2_imshow(gauss_rgb)  # memakai shim cv2_imshow (Colab -> Kaggle)

### Percobaan 2 — Filter Modern: Bilateral Filtering dan Guided Filter

**Bilateral filter** adalah filter non-linear yang menghaluskan citra namun mempertahankan tepi.
Bobot setiap piksel tetangga bergantung pada dua hal sekaligus: (i) **jarak spasial** ke piksel
pusat (seperti Gaussian blur biasa) dan (ii) **kemiripan intensitas warna**. Piksel yang dekat dan
berwarna mirip mendapat bobot besar, sedangkan piksel dengan warna sangat berbeda (misalnya di
seberang tepi) bobotnya ditekan. Karena itu area datar menjadi rata, tetapi garis batas dan detail
tetap terjaga. Kelemahannya adalah lambat karena bobot dihitung per piksel untuk setiap tetangga.

**Guided filter** adalah pendekatan yang lebih baru dan efisien. Diasumsikan bahwa di dalam jendela
lokal, keluaran $q$ merupakan fungsi linear dari citra pemandu (*guide*) $I$:

$$q_i = a_k I_i + b_k, \quad \forall i \in \omega_k$$

Koefisien $a_k$ dan $b_k$ dihitung dari rata-rata lokal ($\mu$), varians ($\sigma^2$), dan kovarians
citra masukan–pemandu. Jika citra masukan = citra pemandu, guided filter berperan mirip bilateral
filter tetapi jauh lebih cepat dan lebih halus (tidak ada *gradient reversal*). Karena pemandu bisa
berupa citra lain, filter ini dipakai untuk HDR tone mapping, image matting, feathering, dan
pemurnian depth map.

In [ ]:
img = cv.imread(os.path.join(WEEK, 'camera.jpg'))
assert img is not None

# --- Bilateral filter (library) ---
bilateral = cv.bilateralFilter(img, d=15, sigmaColor=80, sigmaSpace=80)

# --- Guided filter (implementasi berdasarkan rumus pada penjelasan di atas) ---
def guided_filter(I, p, r=8, eps=1e-3):
    """Guided filter untuk citra 2D (satu kanal). I = guide, p = input."""
    I = I.astype(np.float32)
    p = p.astype(np.float32)
    ksize = (r, r)
    mean_I  = cv.boxFilter(I, -1, ksize)
    mean_p  = cv.boxFilter(p, -1, ksize)
    mean_Ip = cv.boxFilter(I * p, -1, ksize)
    cov_Ip  = mean_Ip - mean_I * mean_p
    mean_II = cv.boxFilter(I * I, -1, ksize)
    var_I   = mean_II - mean_I * mean_I
    a = cov_Ip / (var_I + eps)
    b = mean_p - a * mean_I
    mean_a = cv.boxFilter(a, -1, ksize)
    mean_b = cv.boxFilter(b, -1, ksize)
    return mean_a * I + mean_b

b, g, r = cv.split(img.astype(np.float32) / 255.0)
gb = guided_filter(b, b, r=8, eps=1e-3)
gg = guided_filter(g, g, r=8, eps=1e-3)
gr = guided_filter(r, r, r=8, eps=1e-3)
guided = (np.clip(cv.merge([gb, gg, gr]), 0, 1) * 255).astype(np.uint8)

show_side_by_side([img, bilateral, guided],
                  ['Asli', 'Bilateral Filter', 'Guided Filter'],
                  figsize=(16, 5))
print('waktu rata-rata: bilateral jauh lebih lambat karena bobot non-linear per piksel,')
print('sedangkan guided filter hanya memakai box filter (O(1) per piksel).')

### Percobaan 3 — Feature Map pada CNN (Convolutional Neural Network)

Pada CNN, lapisan konvolusi memakai banyak kernel yang **dipelajari** (bukan ditentukan manual).
Setiap kernel menghasilkan satu *feature map* yang merespons pola tertentu (tepi, sudut, tekstur).
Sel di bawah mensimulasikan satu lapisan konvolusi: beberapa kernel acak dikonvolusikan ke citra,
lalu diberi aktivasi ReLU dan *pooling*.

In [ ]:
gray_cam = cv.cvtColor(img, cv.COLOR_BGR2GRAY).astype(np.float32)
gray_cam = cv.resize(gray_cam, (160, 160))

def conv_layer(img, n_filters=8, ksize=3, seed=None):
    """Simulasi 1 lapisan konvolusi: kernel acak -> konvolusi -> ReLU -> max-pooling 2x2."""
    rng = np.random.default_rng(seed)
    maps = []
    for _ in range(n_filters):
        k = rng.standard_normal((ksize, ksize)).astype(np.float32)
        f = cv.filter2D(img, -1, k)
        f = np.maximum(f, 0)                  # ReLU
        f = cv.resize(f, (f.shape[1] // 2, f.shape[0] // 2))  # pooling
        maps.append(f)
    return maps

show_grid([gray_cam] + conv_layer(gray_cam, seed=1),
          ['Input'] + [f'Filter {i+1}' for i in range(8)], ncols=3, figsize=(11, 11))

# jalankan lagi dengan kernel berbeda -> hasil berbeda (karena kernel acak)
show_grid(conv_layer(gray_cam, seed=2),
          [f'Run-2 Filter {i+1}' for i in range(8)], ncols=4, figsize=(14, 7))

## Jawaban — Percobaan 3 (Feature Map CNN)

Setiap kali kode dijalankan dengan *seed* berbeda, pola feature map yang muncul **berubah** karena
kernel-nya masih acak. Beberapa kernel menonjolkan tepi, sebagian lain menonjolkan area gelap/terang
atau tekstur tertentu — persis seperti feature map pada CNN. Kesimpulan:

1. **Setiap kernel = satu "detektor pola".** Keluaran satu lapisan konvolusi adalah kumpulan feature
   map, satu untuk tiap filter. Filter dengan koefisien menyerupai pendeteksi tepi akan menghasilkan
   peta yang menyoroti tepi, filter lain menyoroti tekstur atau bidang warna.
2. **Kernel acak menghasilkan fitur yang tidak bermakna secara konsisten.** Karena nilainya diambil
   dari distribusi acak, output berubah tiap kali dijalankan dan tidak stabil. Pada CNN, kernel
   **dipelajari** lewat *backpropagation* sehingga akhirnya berisi pola yang berguna (tepi pada
   layer awal, bentuk/semantik pada layer dalam) dan hasilnya **deterministik** untuk bobot yang sama.
3. **ReLU dan pooling** membuat representasi lebih tahan posisi: nilai negatif dihapus dan resolusi
   spasial dikurangi, sehingga fitur yang sama dikenali meski sedikit bergeser.
4. Jumlah parameter yang besar (banyak filter × ukuran kernel) adalah alasan CNN kuat namun butuh
   data dan komputasi besar untuk melatihnya.

### Percobaan 4 — Filter Beauty dan Vintage

Kedua efek ini adalah **kombinasi filter tradisional** (seperti yang lazim dipakai aplikasi kamera
populer). *Beauty* = bilateral filter (melembutkan kulit sambil menjaga tepi) + *unsharp mask*
(mengembalikan sedikit detail). *Vintage* = penurunan kontras, *color grading* sepia hangat, dan
efek *vignette* (sudut gambar menggelap).

In [ ]:
img = cv.imread(os.path.join(WEEK, 'camera.jpg'))

# ---- Beauty: bilateral (halus, tepi tetap) + unsharp mask ----
beauty = cv.bilateralFilter(img, d=15, sigmaColor=80, sigmaSpace=80)
blur_b = cv.GaussianBlur(beauty, (0, 0), 3)
beauty = cv.addWeighted(beauty, 1.6, blur_b, -0.6, 0)   # unsharp mask

# ---- Vintage: turunkan kontras + sepia + vignette ----
low_contrast = cv.addWeighted(img, 0.85, np.zeros_like(img), 0, 20)

sepia_m = np.array([[0.131, 0.534, 0.272],   # baris B, G, R  (BGR)
                    [0.168, 0.686, 0.349],
                    [0.189, 0.769, 0.393]], np.float32)
sepia = cv.transform(low_contrast, sepia_m)

h, w = img.shape[:2]
yy, xx = np.mgrid[0:h, 0:w]
d = np.sqrt((xx - w/2)**2 + (yy - h/2)**2) / np.sqrt((w/2)**2 + (h/2)**2)
vignette = np.clip(1.0 - 0.65 * d**2, 0, 1)[..., None]
vintage = (sepia.astype(np.float32) * vignette).astype(np.uint8)

show_side_by_side([img, beauty, vintage],
                  ['Asli', 'Efek Beauty', 'Efek Vintage'], figsize=(16, 5))

### Percobaan 5 — Filter Anime / Cartoon

Efek kartun dibuat dari kombinasi filter tradisional: **penghalusan warna** (bilateral/median) untuk
membuat bidang warna rata, lalu **penebalan tepi** agar tampak seperti garis tinta gambar kartun.
Di sini dipakai `cv.stylization` (non-photorealistic rendering OpenCV) yang merupakan kombinasi
edge-preserving smoothing + penekanan detail, dilanjutkan `cv.detailEnhance` untuk menegaskan garis.

In [ ]:
img = cv.imread(os.path.join(WEEK, 'peppers.jpg'))

cartoon = cv.stylization(img, sigma_s=60, sigma_r=0.45)
cartoon = cv.detailEnhance(cartoon, sigma_s=10, sigma_r=0.15)

show_side_by_side([img, cartoon], ['Asli', 'Efek Cartoon / Anime'], figsize=(13, 5))

### Percobaan 6 — Filter Malam (Night Filter)

Filter malam meniru tampilan foto malam: **kecerahan diturunkan**, kontras dinaikkan, warna
digerakkan ke arah **biru/cyan** (khas cahaya bulan/tlampu kota), lalu ditambahkan **glow** dengan
memblend hasil blur Gaussian secara *screen/add* agar sumber cahaya tampak berpendar.

In [ ]:
img = cv.imread(os.path.join(WEEK, 'camera.jpg'))

night = cv.convertScaleAbs(img, alpha=0.75, beta=-15)     # gelapkan
b, g, r = cv.split(night.astype(np.float32))
b *= 1.30; g *= 1.05; r *= 0.85                            # tint biru/cyan
night = np.clip(cv.merge([b, g, r]), 0, 255).astype(np.uint8)

glow = cv.GaussianBlur(night, (0, 0), 10)                 # pendar cahaya
night = cv.addWeighted(night, 0.80, glow, 0.40, 0)
night = cv.convertScaleAbs(night, alpha=1.10, beta=-10)   # kontras akhir

show_side_by_side([img, night], ['Asli', 'Efek Malam (Night)'], figsize=(13, 5))

### Percobaan 7 — Filter Pagi dan Efek Kabut

Filter pagi meniru cahaya pagi: kecerahan dinaikkan, kontras sedikit dilembutkan, dan warna
digerakkan ke arah **hangat (kuning/merah)**. Efek kabut dibuat dengan mencampur citra dengan
lapisan putih yang kekuatannya **bertambah ke arah atas** (gradien vertikal, meniru kabut yang
menebal di kejauhan) lalu diblur agar transisinya halus.

In [ ]:
img = cv.imread(os.path.join(WEEK, 'camera.jpg'))

# ---- Filter Pagi ----
morning = cv.convertScaleAbs(img, alpha=1.15, beta=25)
b, g, r = cv.split(morning.astype(np.float32))
r *= 1.12; g *= 1.05; b *= 0.92                            # tint hangat
morning = np.clip(cv.merge([b, g, r]), 0, 255).astype(np.uint8)
morning = cv.GaussianBlur(morning, (0, 0), 1.2)            # cahaya lembut

# ---- Pagi + Kabut ----
h, w = morning.shape[:2]
grad = np.linspace(0.10, 0.75, h, dtype=np.float32)[:, None, None]  # makin ke atas makin tebal
white = np.full_like(morning, 255, dtype=np.float32)
fog = (morning.astype(np.float32) * (1 - grad) + white * grad).astype(np.uint8)
fog = cv.GaussianBlur(fog, (0, 0), 4)

show_side_by_side([img, morning, fog],
                  ['Asli', 'Filter Pagi (Morning)', 'Pagi + Efek Kabut'], figsize=(16, 5))

---
## Tugas Praktikum

### Tugas 1 — Analisis: Mean Filter (Blur Biasa) vs Median Filter pada Noise "Salt and Pepper"

Noise *salt and pepper* menambahkan piksel acak bernilai 0 (**pepper/hitam**) dan 255 (**salt/putih**)
ke citra. Di bawah ini noise dibuat **manual** dengan NumPy (bukan fungsi library), lalu citra
bernoise diolah dengan Mean Filter 3×3 (`cv.blur`) dan Median Filter 3×3 (`cv.medianBlur`).

In [ ]:
def salt_pepper(img, amount=0.05, seed=42):
    """Menambahkan noise salt & pepper secara manual (tanpa library)."""
    rng = np.random.default_rng(seed)
    out = img.copy()
    r = rng.random(img.shape)
    out[r < amount / 2] = 0          # pepper (hitam)
    out[r > 1 - amount / 2] = 255    # salt (putih)
    return out

gray_img = cv.cvtColor(cv.imread(os.path.join(WEEK, 'camera.jpg')), cv.COLOR_BGR2GRAY)
noisy = salt_pepper(gray_img, amount=0.05)

mean3   = cv.blur(noisy, (3, 3))
median3 = cv.medianBlur(noisy, 3)

show_side_by_side([gray_img, noisy, mean3, median3],
                  ['Asli (grayscale)', 'Salt & Pepper 5%',
                   'Mean Filter 3x3', 'Median Filter 3x3'], figsize=(18, 5))

def mse(a, b):
    return float(np.mean((a.astype(np.float64) - b.astype(np.float64)) ** 2))

print('MSE terhadap citra asli (semakin kecil semakin baik):')
print('  citra bernoise   :', round(mse(gray_img, noisy), 2))
print('  Mean Filter 3x3  :', round(mse(gray_img, mean3), 2))
print('  Median Filter 3x3:', round(mse(gray_img, median3), 2))

## Jawaban — Tugas 1 (Mean vs Median pada Salt & Pepper)

**Mengapa Median jauh lebih bersih?** Noise salt & pepper bersifat **impulsif**: nilai piksel
melonjak ekstrem ke 0 atau 255. Kedua filter memakai jendela 3×3 yang sama, tetapi cara
meringkasnya berbeda:

- **Mean filter** menghitung **rata-rata** semua piksel di jendela. Nilai impuls (0 atau 255) ikut
  dijumlahkan sehingga tetap **terbawa** ke hasil. Satu piksel putih di antara tetangga gelap
  (misal 20-an) akan menaikkan rata-rata sekitar $(255 + 8\cdot20)/9 \approx 46$ — bintik tetap
  terlihat, hanya menyebar sebagai kabut/pelunakan. Jadi mean filter mengubah impuls menjadi bercak
  yang lebih besar dan sekaligus mengaburkan tepi.
- **Median filter** mengambil **nilai tengah** setelah mengurutkan 9 nilai jendela. Karena impuls
  hanya 1 dari 9 nilai, ia selalu berada di ujung urutan (paling kecil/paling besar) dan **tidak
  pernah** menjadi nilai median. Median mendekati nilai mayoritas tetangga, sehingga bintik
  hitam/putih benar-benar **hilang** tanpa mengaburkan tepi.

Hasil numerik mendukung: MSE Median Filter terhadap citra asli jauh lebih kecil daripada Mean
Filter. Kesimpulan: untuk noise impulsif (*salt & pepper*) **median filter adalah pilihan yang tepat**,
sedangkan mean filter lebih cocok untuk noise Gaussian.

### Tugas 2 — Evaluasi: Pemilihan Ukuran Kernel Terbaik untuk Sharpening

Dua kernel penajaman diuji pada citra yang sedikit buram (dihasilkan dengan Gaussian blur):

- **3×3** (high-boost klasik, *identity + Laplacian*):
  $\begin{bmatrix}0&-1&0\\-1&5&-1\\0&-1&0\end{bmatrix}$
- **5×5** (*unsharp mask*): $I + \alpha\,(I - \mathrm{box}_{5\times5})$ dengan $\alpha=2$, yaitu
  pusat $1+\alpha(1-\tfrac{1}{25}) = 2{,}92$ dan 24 elemen lainnya $-\alpha/25 = -0{,}08$.
  Jumlah koefisiennya = 1 sehingga kecerahan rata-rata citra tidak berubah.

Metrik yang dipakai:

- **Ketajaman (Tenengrad)** — rata-rata kuadrat gradien Sobel $\overline{g_x^2+g_y^2}$; semakin
  besar semakin tajam.
- **Noise/artefak (hf_noise)** — simpangan baku komponen frekuensi tinggi (selisih citra terhadap
  versi Gaussian-blur-nya); semakin besar semakin banyak noise/halo yang ikut terangkat.

In [ ]:
base = cv.cvtColor(cv.imread(os.path.join(WEEK, 'camera.jpg')), cv.COLOR_BGR2GRAY)
blurry = cv.GaussianBlur(base, (0, 0), 3.0)          # citra sedikit buram

# kernel 3x3 : identity + Laplacian (high-boost)
k3 = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], np.float64)

# kernel 5x5 : unsharp mask I + alpha*(I - box5), alpha = 2, jumlah koefisien = 1
alpha = 2.0
k5 = np.full((5, 5), -alpha / 25.0, np.float64)
k5[2, 2] += 1.0 + alpha
print('jumlah koefisien k3 =', k3.sum(), '| k5 =', k5.sum())

sharp3 = to_uint8(convolve2d(blurry, k3, padding='replicate'))
sharp5 = to_uint8(convolve2d(blurry, k5, padding='replicate'))

show_grid([blurry, sharp3, sharp5],
          ['Blurry (input)', 'Sharpen 3x3', 'Sharpen 5x5'], ncols=3, figsize=(14, 5))

# ketajaman = rata-rata kuadrat gradien Sobel (metrik Tenengrad)
def tenengrad(img):
    f = img.astype(np.float32)
    gx = cv.Sobel(f, cv.CV_32F, 1, 0, ksize=3)
    gy = cv.Sobel(f, cv.CV_32F, 0, 1, ksize=3)
    return float(np.mean(gx**2 + gy**2))

# noise/artefak = energi frekuensi tinggi yang ikut terangkat
def hf_noise(img):
    lo = cv.GaussianBlur(img, (0, 0), 2).astype(np.float64)
    return float(np.std(img.astype(np.float64) - lo))

print(f"{'citra':14s} {'ketajaman (Tenengrad)':>22s} {'hf_noise':>10s}")
for nama, im in [('blurry', blurry), ('sharpen 3x3', sharp3), ('sharpen 5x5', sharp5)]:
    print(f'{nama:14s} {tenengrad(im):22.1f} {hf_noise(im):10.2f}')

## Jawaban — Tugas 2 (Sharpening 3×3 vs 5×5)

- **Kernel 3×3** memperkuat tepi dengan radius hanya 1 piksel. Hasilnya tepi lebih jelas, tetapi
  efeknya halus dan noise yang ikut terangkat relatif sedikit — ketajaman (varians Laplacian) naik
  moderat dan `hf_noise` bertambah sedikit. Hasilnya terlihat natural.
- **Kernel 5×5** memperkuat tepi dengan radius 2 piksel sehingga detail yang diperkuat lebih besar
  dan ketajaman tertinggi, **tetapi** noise dan artefak *halo* (garis terang/gelap di sekitar tepi)
  juga ikut naik paling banyak. Karena jendelanya lebih lebar, ia juga lebih rentan memunculkan
  riak pada area datar.

**Kesimpulan (kernel paling optimal sehari-hari):** **3×3**. Pada citra yang hanya *sedikit* buram,
kernel 3×3 memberi peningkatan ketajaman yang cukup dengan penambahan noise paling kecil sehingga
hasilnya tetap natural. Kernel 5×5 baru bermanfaat bila citra sangat buram dan noise rendah; jika
tidak, ia justru memperburuk artefak. Praktik yang baik: perbesar ukuran kernel **secara bertahap**
sambil memantau ketajaman vs noise, dan terapkan denoising ringan sebelum menajamkan bila noise tinggi.

### Tugas 3 — Kreasi: Filter Kartun Sederhana `kartun(image)`

Fungsi `kartun(image)` menggabungkan dua tahap sesuai instruksi:

1. **Penghalusan warna** dengan **Bilateral Filter** (+ median blur) agar bidang warna menjadi rata
   namun tepi tetap terjaga.
2. **Deteksi garis tepi** dengan **Adaptive Thresholding** (garis tebal, khas kartun) — dan sebagai
   pembanding, dengan **Canny Edge** (garis tipis).
3. **Penggabungan** dengan operasi logika **AND** (perkalian piksel): piksel hasil dipertahankan
   hanya jika berada di area non-tepi; area tepi menjadi hitam (garis tinta).

In [ ]:
def kartun(image, d=9, sigma=75, block=9, C=2, metode='adaptive'):
    """Filter kartun: bilateral (warna rata) x deteksi tepi (garis tinta).

    metode : 'adaptive' -> cv.adaptiveThreshold (garis tebal)
             'canny'    -> cv.Canny (garis tipis)
    """
    # 1) penghalusan warna
    color = cv.bilateralFilter(image, d, sigma, sigma)
    color = cv.medianBlur(color, 5)

    # 2) deteksi garis tepi
    gray = cv.cvtColor(image, cv.COLOR_BGR2GRAY)
    gray = cv.medianBlur(gray, 5)
    if metode == 'canny':
        edges = cv.Canny(gray, 60, 160)
        edges = cv.bitwise_not(edges)                 # tepi hitam, latar putih
    else:
        edges = cv.adaptiveThreshold(gray, 255, cv.ADAPTIVE_THRESH_MEAN_C,
                                     cv.THRESH_BINARY, block, C)
    edges = cv.cvtColor(edges, cv.COLOR_GRAY2BGR)

    # 3) gabungkan: warna AND masker tepi (perkalian piksel)
    return cv.bitwise_and(color, edges)

img = cv.imread(os.path.join(WEEK, 'peppers.jpg'))
kartun_adapt = kartun(img, metode='adaptive')
kartun_canny = kartun(img, metode='canny')

show_side_by_side([img, kartun_adapt], ['Asli', 'kartun() - Adaptive Threshold'], figsize=(13, 5))
show_side_by_side([img, kartun_canny], ['Asli', 'kartun() - Canny Edge'], figsize=(13, 5))

## Jawaban — Tugas 3 (Filter Kartun)

- **Bilateral filter** adalah kunci efek kartun: ia meratakan warna di area yang seragam (kulit,
  langit, dinding) tanpa mengaburkan batas objek. `medianBlur` setelahnya membuang bintik sisa
  sehingga bidang warna terlihat seperti *flat color* pada gambar kartun.
- **Adaptive Thresholding** (`ADAPTIVE_THRESH_MEAN_C`) memilih ambang secara lokal, sehingga garis
  tepi tetap muncul baik di area terang maupun gelap — hasilnya garis **tebal dan tegas** seperti
  tinta, namun pada tekstur halus (mis. tekstur permukaan paprika) bisa muncul garis "berisik".
- **Canny** menghasilkan garis **lebih tipis dan bersih** karena memakai histeresis dua ambang,
  tetapi detail tepi lemah kadang hilang. Pilih Canny bila ingin kesan *sketch*, adaptive threshold
  bila ingin kesan *comic/ink*.
- **Penggabungan dengan AND (perkalian piksel)** tepat secara matematis: area tepi bernilai 0
  sehingga menjadi hitam, area non-tepi bernilai 1 (putih) sehingga warna hasil bilateral lolos apa
  adanya. `cv.bitwise_and` melakukan operasi ini per kanal.
- Secara visual, hasil `kartun()` sudah menyerupai efek aplikasi kamera: warna rata + garis tepi
  hitam. Kualitasnya dapat diatur lewat `d`/`sigma` (kekuatan penghalusan) dan `block`/`C`
  (sensitivitas garis).

---
## Kesimpulan Modul 6

1. **Konvolusi** adalah inti pemfilteran spasial: hasil ditentukan sepenuhnya oleh koefisien kernel.
   Implementasi manual dengan nested loop dan tiga opsi padding (`zero`, `replicate`, `same`) berhasil
   mereproduksi hasil `cv.filter2D` (selisih ≈ 0 pada kernel Gaussian 21×21).
2. **Low pass** (Average, Low Pass, Gaussian) menghaluskan/menekan frekuensi tinggi, sedangkan
   **high pass** (High Pass, Sharpen, Emboss, Sobel, Laplacian/Point/Line/Edge Detection) menonjolkan
   tepi, titik, dan garis sesuai orientasi kernel.
3. **Filter library & modern**: `cv.filter2D`, bilateral filter (non-linear, tepi terjaga), dan
   guided filter (lebih cepat & halus) menjadi dasar efek-efek aplikasi populer (beauty, vintage,
   cartoon, night, morning+fog) yang semuanya adalah **kombinasi filter tradisional**.
4. **Feature map CNN** menunjukkan bahwa setiap kernel adalah satu detektor pola; pada CNN kernel
   tersebut **dipelajari**, bukan acak, sehingga hasilnya stabil dan bermakna.
5. Pada tugas: **median filter** mengalahkan mean filter untuk noise salt & pepper, kernel sharpen
   **3×3** lebih optimal sehari-hari dibanding 5×5 (ketajaman vs noise), dan fungsi **`kartun()`**
   berhasil menggabungkan bilateral filter dengan deteksi tepi.